In [1]:
!pip install pyspark
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark = SparkSession.builder.appName("CombinedChallenge").getOrCreate()

In [2]:
from google.colab import files
uploaded = files. upload()

Saving employees.csv to employees.csv
Saving logins.txt to logins.txt
Saving orders.json to orders.json
Saving sales.csv to sales.csv


In [3]:
logins_df = spark.read.text("logins.txt").withColumnRenamed("value", "name")
logins_df.show()

+-----+
| name|
+-----+
|Rahul|
|Sneha|
|Arjun|
|Rahul|
|Priya|
|Sneha|
|Rahul|
|Karan|
|Arjun|
|Sneha|
|Rahul|
| Amit|
|Priya|
|Karan|
|Sneha|
|Rahul|
|Meera|
|Arjun|
|Sneha|
|Rahul|
+-----+
only showing top 20 rows


In [4]:
total_logins = logins_df.count()
print(f"Total login events: {total_logins}")

Total login events: 26


In [5]:
unique_users = logins_df.select("name").distinct()
unique_users.show()

+--------+
|    name|
+--------+
|   Meera|
|   Sneha|
|   Priya|
|   Rahul|
|   Arjun|
|    Amit|
|Arjun   |
|   Karan|
+--------+



In [6]:
user_counts = logins_df.groupBy("name").count().orderBy(F.desc("count"))
user_counts.show()

+--------+-----+
|    name|count|
+--------+-----+
|   Rahul|    7|
|   Sneha|    6|
|   Priya|    3|
|   Arjun|    3|
|   Karan|    3|
|    Amit|    2|
|   Meera|    1|
|Arjun   |    1|
+--------+-----+



In [7]:
top_3_users = user_counts.limit(3)
top_3_users.show()

+-----+-----+
| name|count|
+-----+-----+
|Rahul|    7|
|Sneha|    6|
|Arjun|    3|
+-----+-----+



In [8]:
active_users_plus_4 = user_counts.filter(F.col("count") > 4)
active_users_plus_4.show()

+-----+-----+
| name|count|
+-----+-----+
|Rahul|    7|
|Sneha|    6|
+-----+-----+



exercise 2


In [9]:
employees_df = spark.read.csv("employees.csv", header=True, inferSchema=True)

In [10]:
print(f"Total Employees: {employees_df.count()}")


Total Employees: 20


In [11]:
employees_df.filter(F.col("department") == " IT").show()

+------+----+----------+------+----+
|emp_id|name|department|salary|city|
+------+----+----------+------+----+
+------+----+----------+------+----+



In [12]:
employees_df.filter(F.col("salary") > 75000).show()
employees_df.select(F.avg("salary")).show()

+------+------+----------+------+---------+
|emp_id|  name|department|salary|     city|
+------+------+----------+------+---------+
|     4| Priya|   Finance| 80000|Hyderabad|
|     7| Meera|   Finance| 82000|Bangalore|
|    10|Vikram|   Finance| 90000|    Delhi|
|    13| Divya|        IT| 77000|Hyderabad|
|    14|Sanjay|   Finance| 85000|  Chennai|
|    17| Sonal|   Finance| 88000|   Mumbai|
|    20| Akash|   Finance| 91000|    Delhi|
+------+------+----------+------+---------+

+-----------+
|avg(salary)|
+-----------+
|    71450.0|
+-----------+



In [13]:
employees_df.orderBy(F.desc("salary")).limit(1).show()
employees_df.orderBy(F.asc("salary")).limit(1).show()

+------+-----+----------+------+-----+
|emp_id| name|department|salary| city|
+------+-----+----------+------+-----+
|    20|Akash|   Finance| 91000|Delhi|
+------+-----+----------+------+-----+

+------+-----+----------+------+------+
|emp_id| name|department|salary|  city|
+------+-----+----------+------+------+
|     5|Karan|        IT| 50000|Mumbai|
+------+-----+----------+------+------+



In [14]:
employees_df.groupBy("department").agg(
    F.count("emp_id").alias("emp_count"),
    F.avg("salary").alias("avg_salary")
).show()

+----------+---------+------------------+
|department|emp_count|        avg_salary|
+----------+---------+------------------+
|        HR|        6|60333.333333333336|
|   Finance|        6|           86000.0|
|        IT|        8|           68875.0|
+----------+---------+------------------+



In [17]:
employees_df.groupBy("city").count().show()
employees_df.orderBy(F.desc("salary")).limit(5).show()


+---------+-----+
|     city|count|
+---------+-----+
|Bangalore|    4|
|  Chennai|    4|
|   Mumbai|    3|
|    Delhi|    4|
|Hyderabad|    5|
+---------+-----+

+------+------+----------+------+---------+
|emp_id|  name|department|salary|     city|
+------+------+----------+------+---------+
|    20| Akash|   Finance| 91000|    Delhi|
|    10|Vikram|   Finance| 90000|    Delhi|
|    17| Sonal|   Finance| 88000|   Mumbai|
|    14|Sanjay|   Finance| 85000|  Chennai|
|     7| Meera|   Finance| 82000|Bangalore|
+------+------+----------+------+---------+



In [19]:
sales_df = spark.read.csv("sales.csv", header=True, inferSchema=True)

In [20]:
sales_df = sales_df.withColumn("revenue", F.col("quantity") * F.col("price"))
sales_df.show()

+-------+------+--------+--------+-----+-------+
|sale_id|emp_id| product|quantity|price|revenue|
+-------+------+--------+--------+-----+-------+
|      1|     1|  Laptop|       1|75000|  75000|
|      2|     2|   Mouse|       3|  500|   1500|
|      3|     3|Keyboard|       2| 1500|   3000|
|      4|     1| Monitor|       1|12000|  12000|
|      5|     4|  Laptop|       1|75000|  75000|
|      6|     3|   Mouse|       2|  500|   1000|
|      7|     5|Keyboard|       1| 1500|   1500|
|      8|     1|  Laptop|       1|75000|  75000|
|      9|     6|   Mouse|       4|  500|   2000|
|     10|     7| Monitor|       2|12000|  24000|
|     11|     8|Keyboard|       2| 1500|   3000|
|     12|     9|   Mouse|       3|  500|   1500|
|     13|    10|  Laptop|       1|75000|  75000|
|     14|    11| Monitor|       1|12000|  12000|
|     15|    12|Keyboard|       2| 1500|   3000|
|     16|    13|  Laptop|       1|75000|  75000|
|     17|    14|   Mouse|       3|  500|   1500|
|     18|    15| Mon

In [21]:
total_revenue = sales_df.select(F.sum("revenue")).collect()[0][0]
print(f"Total Revenue: {total_revenue}")

Total Revenue: 529500


In [22]:
product_analysis = sales_df.groupBy("product").agg(
    F.sum("revenue").alias("total_product_revenue"),
    F.sum("quantity").alias("total_quantity")
)
product_analysis.show()

+--------+---------------------+--------------+
| product|total_product_revenue|total_quantity|
+--------+---------------------+--------------+
|  Laptop|               450000|             6|
|   Mouse|                 7500|            15|
|Keyboard|                12000|             8|
| Monitor|                60000|             5|
+--------+---------------------+--------------+



In [23]:
product_analysis.orderBy(F.desc("total_quantity")).limit(1).show()

+-------+---------------------+--------------+
|product|total_product_revenue|total_quantity|
+-------+---------------------+--------------+
|  Mouse|                 7500|            15|
+-------+---------------------+--------------+



In [24]:
product_analysis.filter(F.col("total_product_revenue") > 100000).show()

+-------+---------------------+--------------+
|product|total_product_revenue|total_quantity|
+-------+---------------------+--------------+
| Laptop|               450000|             6|
+-------+---------------------+--------------+



In [25]:
orders_df = spark.read.option("multiLine", "true").json("orders.json")

orders_flat = orders_df.select(F.explode("orders").alias("order")).select("order.*")

In [26]:
print(f"Total Orders: {orders_flat.count()}")
orders_flat.select(F.sum("amount")).show()

Total Orders: 10
+-----------+
|sum(amount)|
+-----------+
|     259500|
+-----------+



In [27]:
orders_flat.groupBy("customer").sum("amount").show()

+--------+-----------+
|customer|sum(amount)|
+--------+-----------+
|   Sneha|      76500|
|   Priya|      77000|
|   Rahul|      88000|
|   Arjun|       6000|
|   Karan|      12000|
+--------+-----------+



In [28]:
orders_flat.groupBy("city").count().show()

+---------+-----+
|     city|count|
+---------+-----+
|Bangalore|    2|
|  Chennai|    2|
|   Mumbai|    1|
|Hyderabad|    5|
+---------+-----+



In [29]:
combined_df = employees_df.join(sales_df, "emp_id", "inner")

In [30]:
emp_revenue = combined_df.groupBy("emp_id", "name").agg(F.sum("revenue").alias("total_revenue"))

In [31]:
emp_revenue.orderBy(F.desc("total_revenue")).limit(5).show()

+------+------+-------------+
|emp_id|  name|total_revenue|
+------+------+-------------+
|     1| Rahul|       162000|
|     4| Priya|        75000|
|    10|Vikram|        75000|
|    17| Sonal|        75000|
|    13| Divya|        75000|
+------+------+-------------+



In [32]:
dept_revenue = combined_df.groupBy("department").agg(F.sum("revenue").alias("dept_total_revenue"))
dept_revenue.orderBy(F.desc("dept_total_revenue")).limit(1).show()

+----------+------------------+
|department|dept_total_revenue|
+----------+------------------+
|        IT|            269500|
+----------+------------------+



In [33]:
emp_revenue.write.csv("final_sales_report.csv", header=True)